<a href="https://colab.research.google.com/github/matashova12345/Application-Deployment/blob/main/Merge20261023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-commerce Data Analysis

## 1. Best Customer
Find the customer who placed the most orders.

## 2. Best `product_name`
Find the product with the highest quantity sold. The table also shows total sales revenue.

**Data source:** `ecommerce_tables.zip`

In [1]:
from google.colab import files
import os
import zipfile
import pandas as pd

uploaded = files.upload()

zip_file_path = next(
    (filename for filename in uploaded if filename.lower().endswith(".zip")),
    None
)

if zip_file_path is None:
    raise FileNotFoundError("Please upload ecommerce_tables.zip.")

extract_dir = "/content/ecommerce_data"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction completed!")

Saving ecommerce_tables.zip to ecommerce_tables.zip
Extraction completed!


In [2]:
customers = pd.read_csv(f"{extract_dir}/customers.csv")
orders = pd.read_csv(f"{extract_dir}/orders.csv")
products = pd.read_csv(f"{extract_dir}/products.csv")
order_items = pd.read_csv(f"{extract_dir}/order_items.csv")

print("Customers")
display(customers.head())

print("Orders")
display(orders.head())

print("Products")
display(products.head())

print("Order Items")
display(order_items.head())

Customers


,customer_id,first_name,last_name,gender,birth_date,city,state,signup_date
0,1,Patricia,Smith,F,1976-12-20,Houston,IL,2023-07-29
1,2,Sarah,Williams,F,1957-11-07,New York,CA,2024-03-23
2,3,Linda,Taylor,M,2005-05-08,Houston,CO,2025-05-09
3,4,Linda,Anderson,F,1955-08-01,Denver,IL,2026-11-30
4,5,Susan,Hernandez,F,1968-12-12,Houston,PA,2023-07-29


Orders


,order_id,customer_id,store_id,order_date
0,1,49,8,2026-03-30
1,2,39,10,2026-03-16
2,3,40,10,2026-09-28
3,4,8,10,2025-04-12
4,5,98,4,2026-10-03


Products


,product_id,product_name,category,price
0,1,Pro Cream,Groceries,412.94
1,2,Classic Blender,Electronics,321.44
2,3,Lite Cream,Home & Kitchen,96.31
3,4,Plus Desk,Beauty,406.02
4,5,Premium Watch,Beauty,223.43


Order Items


,order_item_id,order_id,product_id,quantity
0,1,447,28,1
1,2,362,5,5
2,3,107,18,3
3,4,340,16,5
4,5,193,11,2


In [3]:
customer_orders = customers.merge(
    orders,
    on="customer_id",
    how="inner"
)

customer_order_counts = (
    customer_orders.groupby(
        ["customer_id", "first_name", "last_name"],
        as_index=False
    )
    .agg(order_count=("order_id", "count"))
    .sort_values(["order_count", "customer_id"], ascending=[False, True])
)

best_customer = customer_order_counts.head(1)

print("Best Customer (most orders)")
display(best_customer)

Best Customer (most orders)


,customer_id,first_name,last_name,order_count
64,65,Barbara,Hernandez,13


In [4]:
product_sales = order_items.merge(
    products,
    on="product_id",
    how="inner"
)

product_sales["total_sales"] = product_sales["quantity"] * product_sales["price"]

product_summary = (
    product_sales.groupby("product_name", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("total_sales", "sum")
    )
    .sort_values(
        ["total_quantity", "total_sales"],
        ascending=[False, False]
    )
)

best_product = product_summary.head(1)

print("Best Product (highest quantity sold)")
display(best_product)

Best Product (highest quantity sold)


,product_name,total_quantity,total_sales
20,Pro Bag,280,31849.68


In [5]:
best_product_by_revenue = product_summary.sort_values(
    "total_sales",
    ascending=False
).head(1)

print("Best Product (highest sales revenue)")
display(best_product_by_revenue)

Best Product (highest sales revenue)


,product_name,total_quantity,total_sales
13,Plus Desk,274,122967.32
